# TSTR SDV Dataset A - Diabetes

In [1]:
#import libraries
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import os
print('Libraries imported!!')

Libraries imported!!


In [2]:
#define directory of functions and actual directory
FUNCTIONS_HOME = '../../../functions/evaluation_functions/' #home directory of the project
REAL_DATA_HOME = '../../../data/raw/chap/' #home directory of the project
SYN_DATA_HOME  = '../../../data/processed/chap/' #home directory of the project
FUNCTIONS_DIR = 'EVALUATION FUNCTIONS/UTILITY'
ACTUAL_DIR = os.getcwd()

#change directory to functions directory
os.chdir(FUNCTIONS_HOME + FUNCTIONS_DIR)
#import functions for data labelling analisys
from utility_evaluation import DataPreProcessor
from utility_evaluation import train_evaluate_model

#change directory to actual directory
os.chdir(ACTUAL_DIR)
print('Functions imported!!')

Functions imported!!


## 1. Read data

In [3]:
#read real dataset
train_data = pd.read_csv(SYN_DATA_HOME + '1_Chap_Data_Synthetic_SDV.csv')
categorical_columns = ['group']
for col in categorical_columns :
    train_data[col] = train_data[col].astype('category')
train_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,0.636626,0.196489,0.580205,0.260716,-0.141685,-0.940402,0.478726,0.089044
1,Group0,-3.663480,-0.576449,-0.036454,-0.365500,-1.418683,0.268234,-0.119606,-0.341160
2,Group0,-2.290469,-0.959807,-0.078261,0.315556,0.102967,-0.569566,-0.385519,-0.124791
3,Group0,1.039903,0.654351,-0.537303,0.108847,4.651015,0.032991,-0.121340,0.216337
4,Group0,2.772303,0.458008,-0.591914,0.025155,-3.196961,0.279064,-0.365028,0.306224
...,...,...,...,...,...,...,...,...,...
2712,Group0,2.498367,-0.237746,-0.204875,-0.199771,-1.730213,0.819302,-0.061461,-0.030586
2713,Group0,3.619119,1.925700,-0.673044,0.158926,5.919626,0.635406,-0.143706,0.311389
2714,Group0,-6.744096,0.678333,-0.666009,0.263141,-3.587673,-0.283128,0.356059,0.222930
2715,Group0,-0.396708,-0.782045,0.437572,0.035828,-0.962446,0.931447,-0.230310,-0.120290


In [4]:
#read test data
test_data = pd.read_csv(REAL_DATA_HOME + '1_Chap_Data_Real_Test.csv')
for col in categorical_columns :
    test_data[col] = test_data[col].astype('category')
test_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,-3.053475,-0.177497,-0.010661,0.205118,-2.773102,-0.653440,0.319585,0.017457
1,Group0,-1.375254,0.761393,0.270692,0.140597,-2.157208,-0.297755,0.126143,0.118270
2,Group0,3.681521,1.147414,-0.119372,0.276912,1.452468,0.068735,0.043671,0.074436
3,Group0,0.383070,0.211771,0.374537,0.254718,5.352641,-0.611636,0.344336,0.044627
4,Group0,2.009235,0.765321,0.100557,0.058659,2.842860,0.980271,-0.014527,0.096506
...,...,...,...,...,...,...,...,...,...
674,Group0,1.185955,-0.248240,-0.258972,0.284599,0.816753,0.553623,-0.456024,0.157975
675,Group0,-4.898967,-0.576216,-0.150236,0.069086,-4.450551,-0.192307,0.034382,-0.174072
676,Group0,-3.339095,0.856460,-1.021265,-0.131611,0.639099,0.783467,-0.128578,-0.067689
677,Group0,-3.844234,0.083773,0.334898,-0.210940,-1.259774,0.726944,-0.299066,-0.123455


In [5]:
target = 'group'
#quick look at the breakdown of class values
print('Train data')
print(train_data.shape)
print(train_data.groupby(target).size())
print('#####################################')
print('Test data')
print(test_data.shape)
print(test_data.groupby(target).size())

Train data
(2717, 9)
group
Group0    2717
dtype: int64
#####################################
Test data
(679, 9)
group
Group0    645
Group1     34
dtype: int64


## 2. Pre-process training data

In [6]:
target = 'group'
categorical_columns = []
numerical_columns = train_data.select_dtypes(include=['int64','float64']).columns.tolist()
categories = [np.array(range(2))] if categorical_columns else []

data_preprocessor = DataPreProcessor(categorical_columns, numerical_columns, categories)
x_train = data_preprocessor.preprocess_train_data(train_data.loc[:, train_data.columns != target])
y_train = train_data.loc[:, target]

x_train.shape, y_train.shape

((2717, 8), (2717,))

## 3. Preprocess test data

In [7]:
x_test = data_preprocessor.preprocess_test_data(test_data.loc[:, test_data.columns != target])
y_test = test_data.loc[:, target]
x_test.shape, y_test.shape

((679, 8), (679,))

## 4. Create a dataset to save the results

In [8]:
results = pd.DataFrame(columns = ['model','accuracy','precision','recall','f1'])
results

,model,accuracy,precision,recall,f1


## 4. Train and evaluate Random Forest Classifier

In [9]:
rf_results = train_evaluate_model('RF', x_train, y_train, x_test, y_test)
results = pd.concat([results, rf_results], ignore_index=True)
rf_results

,model,accuracy,precision,recall,f1
0,RF,NaN,NaN,NaN,NaN


## 5. Train and Evaluate KNeighbors Classifier

In [10]:
knn_results = train_evaluate_model('KNN', x_train, y_train, x_test, y_test)
results = pd.concat([results, knn_results], ignore_index=True)
knn_results

,model,accuracy,precision,recall,f1
0,KNN,NaN,NaN,NaN,NaN


## 6. Train and evaluate Decision Tree Classifier

In [11]:
dt_results = train_evaluate_model('DT', x_train, y_train, x_test, y_test)
results = pd.concat([results, dt_results], ignore_index=True)
dt_results

,model,accuracy,precision,recall,f1
0,DT,NaN,NaN,NaN,NaN


## 7. Train and evaluate Support Vector Machines Classifier

only 1 category, cannot run this

In [12]:
print("y_train unique classes:", y_train.unique())
print("y_train value counts:\n", y_train.value_counts())
print("y_train shape:", y_train.shape)

y_train unique classes: ['Group0']
Categories (1, object): ['Group0']
y_train value counts:
 group
Group0    2717
Name: count, dtype: int64
y_train shape: (2717,)


In [13]:
svm_results = train_evaluate_model('SVM', x_train, y_train, x_test, y_test)
results = pd.concat([results, svm_results], ignore_index=True)
svm_results

,model,accuracy,precision,recall,f1
0,SVM,NaN,NaN,NaN,NaN


## 8. Train and evaluate Multilayer Perceptron Classifier

In [14]:
mlp_results = train_evaluate_model('MLP', x_train, y_train, x_test, y_test)
results = pd.concat([results, mlp_results], ignore_index=True)
mlp_results

,model,accuracy,precision,recall,f1
0,MLP,NaN,NaN,NaN,NaN


## 9. Save results file

In [15]:
results.to_csv('RESULTS/models_results_sdv.csv', index=False)
results

,model,accuracy,precision,recall,f1
0,RF,NaN,NaN,NaN,NaN
1,KNN,NaN,NaN,NaN,NaN
2,DT,NaN,NaN,NaN,NaN
3,SVM,NaN,NaN,NaN,NaN
4,MLP,NaN,NaN,NaN,NaN
